# Exercise and Fitness Data

https://www.kaggle.com/datasets/aakashjoshi123/exercise-and-fitness-metrics-dataset

Focus: In this hands-on activity, we will work with data from exercise and fitness. The dataset is already cleaned. Make at least 3 questions for the data and answer with plots and visualization.

Extra: Incorporate, besides the plots, hypothesis testing from the last class.

You will have 30 minutes to do the activity, then we will discuss your findings and give some ideas.


In [1]:
import pandas as pd

In [13]:
df = pd.read_csv("/content/drive/MyDrive/Elogroup/CNH data literacy/Beg/M7/gym_members_exercise_tracking.csv")

In [14]:
df.head()

,Age,Gender,Weight (kg),Height (m),Max_BPM,Avg_BPM,Resting_BPM,Session_Duration (hours),Calories_Burned,Workout_Type,Fat_Percentage,Water_Intake (liters),Workout_Frequency (days/week),Experience_Level,BMI
0,56,Male,88.3,1.71,180,157,60,1.69,1313.0,Yoga,12.6,3.5,4,3,30.20
1,46,Female,74.9,1.53,179,151,66,1.30,883.0,HIIT,33.9,2.1,4,2,32.00
2,32,Female,68.1,1.66,167,122,54,1.11,677.0,Cardio,33.4,2.3,4,2,24.71
3,25,Male,53.2,1.70,190,164,56,0.59,532.0,Strength,28.8,2.1,3,1,18.41
4,38,Male,46.1,1.79,188,158,68,0.64,556.0,Strength,29.2,2.8,3,1,14.39


In [16]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 973 entries, 0 to 972
Data columns (total 15 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   Age                            973 non-null    int64  
 1   Gender                         973 non-null    object 
 2   Weight (kg)                    973 non-null    float64
 3   Height (m)                     973 non-null    float64
 4   Max_BPM                        973 non-null    int64  
 5   Avg_BPM                        973 non-null    int64  
 6   Resting_BPM                    973 non-null    int64  
 7   Session_Duration (hours)       973 non-null    float64
 8   Calories_Burned                973 non-null    float64
 9   Workout_Type                   973 non-null    object 
 10  Fat_Percentage                 973 non-null    float64
 11  Water_Intake (liters)          973 non-null    float64
 12  Workout_Frequency (days/week)  973 non-null    int

In [15]:
df.columns

Index(['Age', 'Gender', 'Weight (kg)', 'Height (m)', 'Max_BPM', 'Avg_BPM',
       'Resting_BPM', 'Session_Duration (hours)', 'Calories_Burned',
       'Workout_Type', 'Fat_Percentage', 'Water_Intake (liters)',
       'Workout_Frequency (days/week)', 'Experience_Level', 'BMI'],
      dtype='object')

In [9]:
import plotly.express as px

## 1. Which workout type has higher average session duration?

In [22]:
workout_duration = (
    df.groupby("Workout_Type", as_index=False)["Session_Duration (hours)"]
    .mean()
    .sort_values("Session_Duration (hours)", ascending=False)
)

px.bar(
    workout_duration,
    x="Workout_Type",
    y="Session_Duration (hours)",
    title="Average Session Duration by Workout Type"
).show()

The slightly higher average appears to be HIIT, followed closely by Yoga and Strength. Cardio has the lowest average session duration, but the difference is small enough that it may not be practically important.

What is interesting here is that this may go against intuition. We might expect HIIT to be much shorter than other workouts, because high-intensity training is often associated with shorter sessions. However, in this dataset, HIIT has the highest average duration.

## 2. How is Water Intake related to fat percentage

In [44]:
px.scatter(
    df,
    x="Water_Intake (liters)",
    y="Fat_Percentage",
    color="Gender",
    hover_data=["Age", "Workout_Type"],
    title="BMI vs Fat Percentage"
).show()

In [39]:
df.corr(numeric_only=True)["Water_Intake (liters)"].sort_values(ascending=False)

,Water_Intake (liters)
Water_Intake (liters),1.000000
Weight (kg),0.394276
Height (m),0.393533
Calories_Burned,0.356931
Experience_Level,0.304104
Session_Duration (hours),0.283411
Workout_Frequency (days/week),0.238563
BMI,0.213697
Age,0.041528
Max_BPM,0.031621


At first glance, it may look like there is a relationship: people with higher water intake seem to have lower fat percentage. However, the pattern is strongly influenced by gender separation.

The red points, representing female participants, are mostly concentrated at lower water intake values and higher fat percentage values. The blue points, representing male participants, are mostly concentrated at higher water intake values and lower fat percentage values.

So the apparent correlation is probably not only about water intake. It seems to be strongly connected to the fact that gender groups occupy different regions of the plot.

## 3. Does experience level affect Session_Duration (hours)?

In [43]:
px.violin(
    df,
    x="Experience_Level",
    y="Session_Duration (hours)",
    color="Experience_Level",
    box=True,
    title="Session_Duration (hours) by Experience Level"
).show()

The pattern is very clear: as experience level increases, session duration also increases. Participants at level 1 have shorter and more variable sessions, while participants at level 3 have longer sessions concentrated around higher values.

Level 1 is the most spread out group. This suggests that beginners have less consistent workout duration: some train for short sessions, while others train for longer periods.

## 4. Which workout types are the most common?

In [31]:
px.histogram(
    df,
    x="Workout_Type",
    title="Workout Type Count"
).show()

## 5. How are BMI values distributed?

In [32]:
px.histogram(
    df,
    x="BMI",
    nbins=30,
    title="BMI Distribution"
).show()

## Hypothesis test

Question: Do advanced gym members burn more calories than beginners?

We compare the average `Calories_Burned` between two groups:

- Beginner group: `Experience_Level = 1`
- Advanced group: `Experience_Level = 3`

Hypotheses:

- H0: Advanced members burn the same or fewer calories than beginners.
- H1: Advanced members burn more calories than beginners.

This is a one-tailed independent t-test because we are comparing the means of two independent groups and testing whether one group has a higher mean.

In [45]:
from scipy import stats

beginner = df[df["Experience_Level"] == 1]["Calories_Burned"]
advanced = df[df["Experience_Level"] == 3]["Calories_Burned"]

t_stat, p_value_two_tailed = stats.ttest_ind(
    advanced,
    beginner,
    equal_var=False
)

p_value_one_tailed = p_value_two_tailed / 2

print("Beginner mean:", beginner.mean())
print("Advanced mean:", advanced.mean())
print("t-statistic:", t_stat)
print("one-tailed p-value:", p_value_one_tailed)

alpha = 0.05

if (p_value_one_tailed < alpha) and (advanced.mean() > beginner.mean()):
    print("Reject H0: advanced members burn significantly more calories.")
else:
    print("Fail to reject H0: there is not enough evidence that advanced members burn more calories.")

Beginner mean: 726.375
Advanced mean: 1265.3403141361257
t-statistic: 30.118730576074277
one-tailed p-value: 1.501514450880241e-110
Reject H0: advanced members burn significantly more calories.
